# Week 4 — Fine-tune TrOCR + evaluate with CER/WER

**Before running any code this week: Runtime > Change runtime type > GPU.**

This notebook drives `src/train.py` and `src/evaluate.py`. See `src/train.py`'s
docstring for the full list of changes vs. the handout's Cells 1-3 (Seq2SeqTrainer
with `predict_with_generate`, cosine LR + warmup, more epochs with early
stopping on CER, mixed precision, beam-search decoding). The short version:
these changes exist because they measurably reduce Character Error Rate, not
just to make the code longer.


In [ ]:
# Run once per Colab session.
# If you opened this notebook from GitHub (File > Save a copy in GitHub, per Week 1),
# clone your repo so `src/` and the bundled fonts/corpus are available:

import os
if not os.path.exists('src'):
    # Replace with your own fork's URL if you're running this outside the
    # original solution repo.
    !git clone https://github.com/Rameesha-Malik/ML-SI-Solution-Urdu-OCR.git repo
    %cd repo

!pip install -q -r requirements.txt
import sys
sys.path.insert(0, 'src')
print('Setup complete.')


In [ ]:
import torch
print('Using device:', 'cuda' if torch.cuda.is_available() else 'cpu')
if not torch.cuda.is_available():
    print('WARNING: No GPU detected. Go to Runtime > Change runtime type > GPU.')


## Train

Adjust `--epochs` / `--batch-size` for your Colab tier. Training logs CER/WER after every epoch (unlike the handout's loop, which only prints training loss).

In [ ]:
!python src/train.py \
    --train-csv data/train.csv --test-csv data/test.csv \
    --output-dir /content/drive/MyDrive/SI26-urdu-ocr-model \
    --epochs 12 --batch-size 8 --grad-accum 2


(If you'd rather keep checkpoints local to the Colab VM instead of Drive, drop the `/content/drive/...` path and use `models/urdu-trocr` — just remember to save to Drive/HF Hub before the session ends.)

## Evaluate

Reports Character Error Rate, Word Error Rate, and exact-match accuracy (the handout's original metric) side by side, plus the 5 worst predictions for your write-up.

In [ ]:
!python src/evaluate.py /content/drive/MyDrive/SI26-urdu-ocr-model/final \
    --test-csv data/test.csv \
    --report-csv reports/eval_report.csv


**What to record** (same ask as the handout, now with real numbers instead of just exact-match): your CER / WER / exact-match accuracy, and 3-5 wrong examples from `reports/eval_report.csv` (sort by the `cer` column). These go into your Week 5 README's Results section.

## Save your model

Already saved to Drive by `--output-dir` above during training. To also publish it to the HuggingFace Hub (recommended — Spaces deployment in Week 5 can then load it with `MODEL_PATH=yourusername/your-model-name`, no Drive mount needed):

In [ ]:
from huggingface_hub import notebook_login
notebook_login()  # paste a HuggingFace token with write access

from transformers import TrOCRProcessor, VisionEncoderDecoderModel

model_dir = '/content/drive/MyDrive/SI26-urdu-ocr-model/final'
repo_id = 'yourusername/urdu-trocr-si26'  # <-- change this

model = VisionEncoderDecoderModel.from_pretrained(model_dir)
processor = TrOCRProcessor.from_pretrained(model_dir)
model.push_to_hub(repo_id)
processor.push_to_hub(repo_id)
print(f'Model published at: https://huggingface.co/{repo_id}')
